## ORv01 Visium v1: Preprocessing & Initial Analysis

Date: 09/30/25

Analysis of Visium v1 data for Orv-01 (D10resect), including initial preprocessing and cell clustering.

### Set up

In [ ]:
# Import libs
from pathlib import Path
from sklearn.metrics import silhouette_samples, silhouette_score
import os

import scanpy as sc
import squidpy as sq
import anndata as ad
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

sc.logging.print_header()
print(f"squidpy=={sq.__version__}")

In [ ]:
# Set paths 
data_dir = '' # canFam4 data
out_dir = ''
obj_dir = ''

In [ ]:
# Create subdirs
os.makedirs(f"{out_dir}/Preprocessing", exist_ok=True)

In [ ]:
# Set current figure path and settings
fig_dir = out_dir + 'Preprocessing/'

plt.rcParams['savefig.dpi'] = 400

In [ ]:
# Read in adata from progress point
adata = sc.read_h5ad(f'{obj_dir}orv01_spatial_clustered.h5ad')
adata

In [ ]:
adata = sc.read_h5ad(f'{obj_dir}orv01_spatial_post_QC.h5ad')
adata

### Initial overview of data

In [ ]:
# Read in img and adata
adata = sq.read.visium(path=data_dir, library_id='orv01')
adata

In [ ]:
# List all samples in adata object
sample_ids = list(adata.uns['spatial'].keys())
sample_ids

#### Inspecting different aspects of the adata object

In [ ]:
adata.obs  # rows are spots, array_row and array_col are x and y spot coords in img

In [ ]:
adata.uns['spatial']

In [ ]:
# Basic plot
sq.pl.spatial_scatter(adata)
plt.savefig(f'{fig_dir}basic_tissue_image.png')

In [ ]:
# Gets you the same plot but via matplotlib
plt.imshow(adata.uns['spatial']['orv01']['images']['hires']) # plotting the hires image
#plt.imshow(adata.uns['spatial']['orv01']['images']['lowres']) # plotting the lowres image

In [ ]:
# Plot the fullres version (plotting spots on top of img)
coords = adata.obsm['spatial'] * adata.uns['spatial']['orv01']['scalefactors']['tissue_lowres_scalef']

plt.imshow(adata.uns['spatial']['orv01']['images']['lowres'])
plt.scatter(coords[:,0], coords[:,1], s=1, c="green")

plt.savefig(f'{fig_dir}basic_tissue_with_spots.png')

In [ ]:
# Cropped to a specific tissue section

# Get spot coordinates
coords = adata.obsm["spatial"]
# Find midpoints
x_mid = np.median(coords[:, 0])
y_mid = np.median(coords[:, 1])
# define crop — left, right, top, bottom
x_min, x_max = coords[:, 0].min(), coords[:, 0].max()
y_min, y_max = coords[:, 1].min(), coords[:, 1].max()

# Plot
crop_coords = (int(x_min), int(x_mid), int(y_mid), int(y_max))
sq.pl.spatial_scatter(adata, crop_coord=crop_coords)

plt.savefig(f'{fig_dir}basic_tissue_cropped.png')

### Performing QC Metrics

In [ ]:
adata.var_names_make_unique()

In [ ]:
mt_genes1 = adata.var_names.str.startswith("chrM-")
mt_genes2 = adata.var_names.str.startswith("MT")
mt_genes3 = adata.var_names.str.startswith("chrMT-")
print(adata.var_names[mt_genes1])
print(adata.var_names[mt_genes2])
print(adata.var_names[mt_genes3])

In [ ]:
# Compute basic QC metrics
sc.pp.calculate_qc_metrics(adata, inplace=True)

# MT fraction
adata.var['mt'] = adata.var_names.str.startswith("MT")
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)

# Ribo fraction
adata.var['ribo'] = adata.var_names.str.startswith(("RPS", "RPL"))
sc.pp.calculate_qc_metrics(adata, qc_vars=['ribo'], inplace=True)

In [ ]:
adata.obs

In [ ]:
# Plot
sc.settings.figdir = f"{out_dir}Preprocessing/"
plt.rcParams['figure.figsize'] = [5, 5] 

sc.pl.violin(adata, keys='n_genes_by_counts', size=1.5,color='skyblue', save='_ngenes_by_counts.png') # num of genes
sc.pl.violin(adata, keys='pct_counts_mt', size=1.5, color='skyblue', save='_pct_counts_mt.png') # pct mt distribution
sc.pl.violin(adata, keys='pct_counts_ribo', size=1.5, color='skyblue', save='_pct_counts_ribo.png') # pct mt distribution

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(20, 4))

axs[0].set_title("Total counts per spot")
sns.histplot(adata.obs["total_counts"], kde=False, ax=axs[0],color='skyblue', edgecolor='darkblue')

axs[1].set_title("Unique transcripts per spot")
sns.histplot(adata.obs["n_genes_by_counts"],kde=False,ax=axs[1],color='skyblue', edgecolor='darkblue')

axs[2].set_title("Percent MT gene expression")
sns.histplot(adata.obs["pct_counts_mt"],kde=False,ax=axs[2],color='skyblue', edgecolor='darkblue',binwidth=1)

axs[3].set_title("Percent Ribo gene expression")
sns.histplot(adata.obs["pct_counts_ribo"],kde=False,ax=axs[3],color='skyblue', edgecolor='darkblue',binwidth=2)

plt.savefig(f'{out_dir}Preprocessing/qc_histograms.png', dpi=400) 

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(20, 4))

axs[0].set_title("Total counts per spot")
sns.histplot(adata.obs["total_counts"], kde=False, ax=axs[0],color='skyblue', edgecolor='darkblue')

axs[1].set_title("Unique transcripts per spot")
sns.histplot(adata.obs["n_genes_by_counts"],kde=False,ax=axs[1],color='skyblue', edgecolor='darkblue')

axs[2].set_title("Percent MT gene expression")
sns.histplot(adata.obs["pct_counts_mt"],kde=False,ax=axs[2],color='skyblue', edgecolor='darkblue',binwidth=1)

axs[3].set_title("Percent Ribo gene expression")
sns.histplot(adata.obs["pct_counts_ribo"],kde=False,ax=axs[3],color='skyblue', edgecolor='darkblue',binwidth=2)

plt.savefig(f'{out_dir}Preprocessing/qc_histograms.png', dpi=400)                                                   

In [ ]:
plt.rcParams['figure.figsize'] = [5, 5]

sc.pl.scatter(adata, x='n_genes_by_counts', y='pct_counts_mt', size=20, color='total_counts', show=False, save='_total_counts.png')
sc.pl.scatter(adata, x='n_genes_by_counts', y='pct_counts_mt', size=20, color='skyblue', show=False, save='_pct_mt.png')
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts', size=20, color='pct_counts_ribo', show=False, save='_scatter_ribo.png')
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts', size=20, color='skyblue', show=False, save='_total_counts_x_ngenes.png')


#### Plotting QC Metrics onto tissue 

In [ ]:
# Plot the fullres version (plotting spots on top of img)
coords = adata.obsm['spatial'] * adata.uns['spatial']['orv01']['scalefactors']['tissue_lowres_scalef']

plt.imshow(adata.uns['spatial']['orv01']['images']['lowres'])
plt.scatter(coords[:,0], coords[:,1], s=1, c="green")

# plt.savefig(f'{fig_dir}basic_tissue_with_spots.png')


In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 4))

sq.pl.spatial_scatter(adata, color='total_counts', ax=axs[0])
sq.pl.spatial_scatter(adata, color='n_genes_by_counts', ax=axs[1])
sq.pl.spatial_scatter(adata, color='pct_counts_ribo', ax=axs[2])

plt.savefig(f'{out_dir}Preprocessing/qc_metrics_on_tissue.png', dpi=400)                                                   

In [ ]:
# Conditions for highlighting
df = adata.obs
condition_mt = df['pct_counts_mt'] > 7.5
condition_ngenes = df['n_genes_by_counts'] < 100

# First Plot: pct_counts_NegPrb vs. n_genes_by_counts
plt.figure(figsize=[5, 5])
plt.scatter(df['pct_counts_mt'], df['n_genes_by_counts'], c='grey', alpha=0.5, s=20)  # Base layer
plt.scatter(df[condition_mt & condition_ngenes]['pct_counts_mt'], 
            df[condition_mt & condition_ngenes]['n_genes_by_counts'], 
            c='red', alpha=0.75, s=20)  # Highlighted cells
plt.title('MT % Counts by Gene Counts')
plt.xlabel('MT Expression (%)')
plt.ylabel('Gene Counts')

plt.savefig(f'{fig_dir}/negprb_by_genecount_scatter.png', dpi=400)
plt.show()

In [ ]:
adata

### Spot filtering based on QC Metrics

The majority of spots display strong sequencing depth, with the exception of a group in the bottom right. % MT expression is generally quite low.

Filtering to keep spots/genes:
- Expressing minimal number of genes (>250) and counts (>500)
- MT % expression < 10 (included in second pass)


Pre-filtering # spots: 4,609


In [ ]:
adata.shape

Fraction of counts assigned to each gene over all cells (pre-filtering):

In [ ]:
sc.settings.figdir = f"{out_dir}Preprocessing/"

ax = sc.pl.highest_expr_genes(adata, n_top=20, show=False)
plt.xlim(0, 35) # reset x axis limits -- removing outliers
plt.savefig(f"{sc.settings.figdir}/highest_expr_genes_pre_filtering.png", dpi=400, bbox_inches='tight')

In [ ]:
adata.obs["n_genes_by_counts"].describe()

Filter spots:

In [ ]:
sc.pp.filter_cells(adata, min_counts=250)
sc.pp.filter_cells(adata, min_genes=250)
adata = adata[adata.obs['pct_counts_mt'] < 12.5, :]
adata

In [ ]:
print(f"= {4609 - 4276} spots filtered out")

In [ ]:
# Checking the result of filtering
sq.pl.spatial_scatter(adata, color='n_genes_by_counts')

#### QC Post-filtering:

In [ ]:
sc.settings.figdir = f"{out_dir}Preprocessing/"

In [ ]:
# Define the features you want to plot
features = ['n_genes_by_counts', 'total_counts', 'pct_counts_mt']

# Number of features
n_features = len(features)

# Create subplots
fig, axs = plt.subplots(1, n_features, figsize=(15, 5))

# Plot each feature in a separate subplot
for i, feature in enumerate(features):
    sc.pl.violin(adata, keys=feature, jitter=0.4, size=0, ax=axs[i], show=False, color='skyblue', edgecolor='black')
    axs[i].set_title(feature)
    
# Adjust layout and save
plt.tight_layout()
plt.savefig(f'{sc.settings.figdir}/qc_violinplots_post_filtering.png')

In [ ]:
# Inspect how the highest expressed genes have changed
sc.pl.highest_expr_genes(adata, n_top=40, save='_post_filtering.png')

In [ ]:
# Calculate % of spots removed -- first pass
(4609 / 4276)/4609 * 100

Approx 2.34% of spots filtered out

In [ ]:
obj_dir

In [ ]:
# Save adata 
adata.write(f'{obj_dir}/orv01_spatial_post_QC.h5ad')

#### Post-filter QC plots

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(20, 4))

axs[0].set_title("Total counts per spot")
sns.histplot(adata.obs["total_counts"], kde=False, ax=axs[0],color='skyblue', edgecolor='darkblue')

axs[1].set_title("Unique transcripts per spot")
sns.histplot(adata.obs["n_genes_by_counts"],kde=False,ax=axs[1],color='skyblue', edgecolor='darkblue')

axs[2].set_title("Percent MT gene expression")
sns.histplot(adata.obs["pct_counts_mt"],kde=False,ax=axs[2],color='skyblue', edgecolor='darkblue',binwidth=0.05)

axs[3].set_title("Percent Ribo gene expression")
sns.histplot(adata.obs["pct_counts_ribo"],kde=False,ax=axs[3],color='skyblue', edgecolor='darkblue',binwidth=2)

plt.savefig(f'{out_dir}Preprocessing/qc_histograms_post_filtering.png', dpi=400)                                                   

In [ ]:
sc.settings.figdir = f"{out_dir}Preprocessing/"

sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts', size=20, color='pct_counts_mt', show=False, save='_post_filter.png')

### Selection of Highly Variable Genes

In [ ]:
adata

In [ ]:
adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata, inplace=True)
sc.pp.log1p(adata)

In [ ]:
adata.layers["log_norm"] = adata.X.copy()

In [ ]:
sc.settings.figdir = f"{out_dir}Clustering/"

sc.pp.highly_variable_genes(adata, n_top_genes=3000)
sc.pl.highly_variable_genes(adata, save='_highly_variable_genes.png')

### PCA and Clustering

In [ ]:
sc.pp.pca(adata)

In [ ]:
plt.rcParams['figure.figsize'] = [6, 6]
sc.pl.pca_variance_ratio(adata, log=True, n_pcs=50)

Using 20 PCs to ensure capture of biological heterogeneity

In [ ]:
fig_dir = out_dir + 'Clustering/'

In [ ]:
# Ensure PCA has been run, variance ratios should be in adata.uns['pca']
variance_ratios = adata.uns['pca']['variance_ratio'][:50]

# Plot the variance ratios for the first 50 PCs
plt.plot(range(1, 51), variance_ratios, marker='o', linestyle='-', color='grey', alpha=0.75)

# Highlight PC20
plt.plot(10, variance_ratios[9], 'ro')  # 'ro' creates a red circle at PC20

# Add enhancements for publication quality
plt.title('PCA Variance Ratios', fontsize=16)
plt.xlabel('Principal Component', fontsize=14)
plt.ylabel('Variance Ratio', fontsize=14)
plt.xticks(ticks=range(0, 51, 5), fontsize=12)  # Show labels for every 5th PC
plt.yticks(fontsize=12)
plt.grid(True, which='both', linestyle='--', linewidth=0.5, color='grey', alpha=0.7)

# Show plot
plt.tight_layout()  # Adjust the layout to make room for the plot elements
plt.savefig(f'{fig_dir}pca_variance_ratios.png')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Extract the PCA variance ratios from the adata object, assuming it's stored in a standard location.
# For the demonstration, let's generate a mock variance_ratio array similar to what you'd find in your adata object.
# Replace this line with your actual data extraction.
#variance_ratios = np.sort(np.random.rand(50))[::-1]  # Assuming descending variance ratios for demonstration
variance_ratios = adata.uns['pca']['variance_ratio'][:50]

# Create the elbow plot
plt.figure(figsize=(6, 6))
pc_numbers = np.arange(len(variance_ratios)) + 1  # PC numbers (1-based indexing)

# Plot all PCs in a default color
plt.plot(pc_numbers, variance_ratios, marker='o', linestyle='-', color='blue', alpha=0.75)

# Highlight PC26 in red
plt.plot(pc_numbers[14], variance_ratios[14], marker='o', linestyle='', color='red', markersize=10)

# Set log scale for the y-axis
plt.yscale('log')

# Enhancements for readability and aesthetics
plt.title('PCA Variance Ratio', fontsize=16)
plt.xlabel('Principal Components', fontsize=14)
plt.ylabel('Variance Ratio (log)', fontsize=14)
plt.xticks(ticks=np.arange(0, 51, 5), fontsize=12)  # Adjust ticks as needed
plt.yticks(fontsize=12)
plt.grid(True, which='both', linestyle='--', linewidth=0.5, color='grey', alpha=0.7)

plt.tight_layout()
plt.savefig(f'{fig_dir}pca_variance_ratios_log.png')
plt.show()

In [ ]:
fig_dir

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Get variance
variance_ratio = adata.uns['pca']['variance_ratio'][:50]
# Calculate cumulative variance explained for better insight
cumulative_variance = np.cumsum(variance_ratio)

# Creating the elbow plot
plt.figure(figsize=(6, 6))
#plt.plot(range(1, 51), variance_ratio, 'o-', color='darkblue', label='Individual')
plt.bar(range(1, 51), variance_ratio, color='blue', label='Individual')
plt.plot(range(1, 51), cumulative_variance, 'o-', color='green', label='Cumulative')

# Enhancements for publication quality
plt.title('Explained Variance Ratio of n PCs', fontsize=16)
plt.xlabel('Number of PCs', fontsize=14)
plt.ylabel('Variance Ratio (log scale)', fontsize=14)
plt.yscale('log')  # Logarithmic scale for the y-axis
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)  # Grid with transparency for less obstruction

# Optional: Highlight the elbow point - adjust based on your analysis
elbow_point = 20 # Example point where you decide the elbow is
plt.axvline(x=elbow_point, linestyle='--', color='red', label='Elbow Point')
plt.axhline(y=variance_ratio[elbow_point-1], linestyle='--', color='grey', alpha=0.7)
plt.text(elbow_point, variance_ratio[elbow_point-1], f'  {elbow_point} PCs', verticalalignment='bottom', fontsize=12)

plt.tight_layout()  # Adjusts subplots to fit into the figure area.
plt.savefig(f'{fig_dir}pca_variance_elbow_plot_final.png', dpi=400)
plt.show()

In [ ]:
sc.pl.pca(adata, color=['log1p_n_genes_by_counts', 'log1p_total_counts'], components=['1,2'], show=False)
plt.tight_layout()
plt.savefig(f'{fig_dir}/scatter_top_two_pcs.png')

In [ ]:
os.chdir(fig_dir)

In [ ]:
os.getcwd()

In [ ]:
os.mkdir('pc_testing')

In [ ]:
# Test different PCs
sc.settings.figdir = f"{out_dir}Clustering/pc_testing/"
plt.rcParams['figure.figsize'] = [5, 5] 

n_pcs = [10,15,20,25]

for i in n_pcs:
    sc.pp.neighbors(adata, n_pcs=i) # Neighbors graph
    sc.tl.umap(adata) # UMAP embeddings
    sc.pl.umap(adata, color='n_genes_by_counts', title=f'{i}pcs', save=f'_{i}pcs.png')

In [ ]:
# Choosing 20
sc.pp.neighbors(adata, n_pcs=20) # Neighbors graph
sc.tl.umap(adata) # UMAP embeddings

In [ ]:
# Clustering at different resolutions
resolutions = [0.05, 0.06, 0.07, 0.08, 0.09, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6]

clusterings = {}

for i in resolutions:
    print('Clustering at resolution:', i, '.....')
    sc.tl.leiden(adata, key_added=f'leiden_{i}', resolution=i)
    clusterings[i] = adata.obs[f'leiden_{i}'].copy()
    print('Done')

# Convert to list
labels = []
for i in clusterings.keys():
    labels.append(f'leiden_{i}')

In [ ]:
# Save adata
adata.write(f'{obj_dir}orv01_spatial_clustered.h5ad')

In [ ]:
labels

In [ ]:
# Removing clusterings that only have 1 group
# Removing 0.05
del adata.obs['leiden_0.05']
del labels[0]
# Removing 0.06
del adata.obs['leiden_0.06']
del labels[0]


In [ ]:
adata.obs['leiden_0.07'].unique()

In [ ]:
# Calcluate ASW scores to benchmark resolutions
print(f'Calculating ASW scores for each resolution...')
scores = {}
for i in labels:
    print(f'Calculating ASW for {i}')
    assignments = adata.obs[i]
    # Calculate score based on X_scvi representation
    asw_score = silhouette_score(adata.obsm['X_umap'], assignments)
    scores[i] = asw_score
    print(asw_score)

print(scores)

In [ ]:
# Plot scores
sc.set_figure_params(scanpy=True, dpi=80, figsize=(6,5), dpi_save=400)
resolutions = [0.07, 0.08, 0.09, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6]

plt.plot(resolutions, scores.values(), marker='o', color='darkblue')
plt.title("ASW Scores by Clustering Resolution")
plt.xlabel("Resolution")
plt.ylabel("ASW Score")
plt.savefig(f'{fig_dir}asw_plot_final.png')
plt.show()

In [ ]:
out_dir

In [ ]:
sc.settings.figdir = f"{out_dir}Clustering/umaps/"
plt.rcParams['figure.figsize'] = [5, 5] 

# Plot umaps
for i in labels:   
    n_clusters = len(adata.obs[i].unique())
    colors = sns.color_palette('husl', n_clusters)
    sc.pl.umap(adata, color=i, palette=colors, size=25, legend_loc='on data', save=f'_{i}.png')

In [ ]:
os.getcwd()

In [ ]:
# Plot clusterings on tissue
os.mkdir('tissue_plots')

fig, axs = plt.subplots(1, 3, figsize=(15, 4))
sq.pl.spatial_scatter(adata, color='leiden_0.1', ax=axs[0])
sq.pl.spatial_scatter(adata, color='leiden_0.2', ax=axs[1])
sq.pl.spatial_scatter(adata, color='leiden_0.3', ax=axs[2])
plt.savefig(f'{out_dir}Clustering/tissue_plots/tissue_plots_1st_three.png', dpi=400)

fig, axs = plt.subplots(1, 3, figsize=(15, 4))
sq.pl.spatial_scatter(adata, color='leiden_0.4', ax=axs[0])
sq.pl.spatial_scatter(adata, color='leiden_0.5', ax=axs[1])
sq.pl.spatial_scatter(adata, color='leiden_0.6', ax=axs[2])
plt.savefig(f'{out_dir}Clustering/tissue_plots/tissue_plots_2nd_three.png', dpi=400)

### Calculating Cluster DEGs 

First, calculating DEGs for a lower resolution to determine broad cell type annotations

Manual annotations will be combined with cell2location (automated) results to determine final cell type identities 


In [ ]:
def perform_de(adata, grouping):
    #calculate total genes
    total_genes = adata.shape[1]

    #calculate marker genes
    sc.tl.rank_genes_groups(adata, groupby=grouping, method='wilcoxon', key_added='rank_genes', n_genes = total_genes)
    sc.tl.filter_rank_genes_groups(adata, key = 'rank_genes', groupby=grouping)
    sc.pl.rank_genes_groups(adata, key='rank_genes', fontsize=12, save=f'ranked_cluster_degs_{grouping}.png')
    
    de  = pd.DataFrame({'genes':[],'logfoldchanges':[],'pvals':[],'cluster':[]})
    for group in set(adata.obs[grouping]):
        d = {'genes': adata.uns['rank_genes']['names'][group],\
         'logfoldchanges': adata.uns['rank_genes']['logfoldchanges'][group],\
         'pvals':adata.uns['rank_genes']['pvals'][group],\
         'pvals_adj':adata.uns['rank_genes']['pvals_adj'][group],\
         'cluster':group}
        de = pd.concat([de,pd.DataFrame(d)])

    de  = pd.DataFrame({'genes':[],'logfoldchanges':[],'pvals':[],'cluster':[]})
    for group in set(adata.obs[grouping]):
        d = {'genes': adata.uns['rank_genes_groups_filtered']['names'][group],\
         'logfoldchanges': adata.uns['rank_genes_groups_filtered']['logfoldchanges'][group],\
         'pvals':adata.uns['rank_genes_groups_filtered']['pvals'][group],\
         'pvals_adj':adata.uns['rank_genes_groups_filtered']['pvals_adj'][group],\
         'cluster':group}
        de = pd.concat([de,pd.DataFrame(d)])

In [ ]:
perform_de(adata, 'leiden_0.2')

In [ ]:
# Supplement annotation by dot plot of cluster DEGS
sc.settings.figdir = fig_dir + 'cluster_degs/'

clustering_resolutions = ['leiden_0.2', 'leiden_0.3', 'leiden_0.4']

for i in clustering_resolutions:
    # Calculate DEGs for 0.2
    sc.tl.rank_genes_groups(
        adata, groupby=i, method="wilcoxon", key_added=f"dea_{i}")
    # Visualize dot plot
    sc.pl.rank_genes_groups_dotplot(
        adata, groupby=i, standard_scale="var", n_genes=8, key=f"dea_{i}",
        save=f'manual_degs_top8_{i}.png'
    )

In [ ]:
adata.uns.keys()

In [ ]:
# Top 10 DEGs per cluster
pd.set_option('display.max_columns', None)
# res 0.2
pd.DataFrame(adata.uns['dea_leiden_0.2']['names']).head(10)


In [ ]:
# res 0.3
pd.DataFrame(adata.uns['dea_leiden_0.3']['names']).head(10)

In [ ]:
# res 0.4
pd.DataFrame(adata.uns['dea_leiden_0.4']['names']).head(10)

In [ ]:
sc.set_figure_params(figsize=(5,5))
sc.settings.figdir = fig_dir + 'marker_genes/'


# Broad cell type markers
cell_markers = {'B_cells': ['CD19', 'CD79A', 'CD79B', 'PAX5', 'JCHAIN'],
                'T_cells': ['CD3E', 'CD3D', 'CD4', 'CD8A'],
                'Myeloid': ['CD68', 'CD163'],
                'Proliferation': ['MKI67', 'TOP2A', 'CENPF'],
                'Other': ['RPS27']}

# Plot
for cell, markers in cell_markers.items():
    sc.pl.umap(
            adata,
            layer='counts',
            color=markers,
            vmin=0,
            #vmax="p90", 
            sort_order=False,  
            frameon=False,
            cmap="Reds",
            save=f'_{cell}_marker_genes_raw_counts.png')

# Plotting log norm expression
for cell, markers in cell_markers.items():
    sc.pl.umap(
            adata,
            layer='log_norm',
            color=markers,
            vmin=0,
            #vmax="p90", 
            sort_order=False,  
            frameon=False,
            cmap="Reds",
            save=f'_{cell}_marker_genes_log_counts.png')

In [ ]:
# Plotting a single gene's expression spatially
sc.set_figure_params(figsize=(5,5))

adata.X = adata.layers['log_norm']
sq.pl.spatial_scatter(adata, color=['CD79A','CD79B'], save='spatial_bcell_markers_log.png')

In [ ]:
sq.pl.spatial_scatter(adata, color=['CD3E','CD4', 'CD8A'], layer='log_norm', save='spatial_tcell_markers_log.png')

#### Plotting c2l cell type predictions

In [ ]:
adata = sc.read_h5ad(f'{obj_dir}orv01_spatial_c2l_annotated.h5ad')

In [ ]:
adata.obs

In [ ]:
sc.pl.umap(adata, color='c2l_predicted')

In [ ]:
sc.pl.umap(adata, color='grouped_annotations')

In [ ]:
adata_c2l = adata.copy()

In [ ]:
adata = sc.read_h5ad(f'{obj_dir}orv01_spatial_clustered.h5ad')

In [ ]:
# Copy selected columns based on matching obs_names
cols_to_copy = ["c2l_predicted", "grouped_annotations"]

adata.obs[cols_to_copy] = adata_c2l.obs.loc[
    adata.obs_names, cols_to_copy
].values

In [ ]:
fig_dir = out_dir + 'Deconvolution/'

In [ ]:
sc.set_figure_params(figsize=(5,5))
sc.settings.figdir = fig_dir

sc.pl.umap(adata, color='grouped_annotations', save='_c2l_grouped_annotations.png')
sc.pl.umap(adata, color='c2l_predicted', save='_c2l_raw_annotations.png')